# CSE475 — Phase 1 explained from zero (no jargon)

This notebook teaches you, in plain English, what the project does and exactly what every file does — line by line. Read top to bottom, run each cell.

**How to use this notebook (home PC):**
1. Open it in **VS Code** (install the Python extension if needed).
2. Top-right: **Select Kernel → Python Environments → `.venv-home-cpu`** (that has torch + everything installed).
3. Run cells in order with the ▶ button. Everything here is **safe to run** — it never touches `results/phase1_baseline.json`, so your Phase 1 golden rule stays intact.

Alternative without VS Code: `pip install jupyter` inside `.venv-home-cpu`, then `.\.venv-home-cpu\Scripts\python.exe -m jupyter lab` and open this file.


## Part 0 — What you're actually building

One sentence: **teach a computer to look at a skin photo and say "acne", "normal", or "rosacea".**

The photos already sit in folders that name the answer:

```
skin_disease_images/
  train/       acne/  normal/  rosacea/    <- photos the brain studies
  validation/  acne/  normal/  rosacea/    <- photos used to check progress (not studied)
  test/        acne/  normal/  rosacea/    <- photos used for the final exam (never touched during study)
```

**The learning loop** — this is the *only* "hard" part, and every image classifier in every paper does it:

```
for each epoch:                  # repeat the whole study session 15 times
    for each batch of 32 photos:  # feed the brain a handful at a time
        guess = model(photo)      # the brain makes a guess
        loss  = how_wrong(guess)  # measure the mistake as a number
        loss.backward()           # figure out which wires caused the mistake
        optimizer.step()          # nudge the wires so the mistake shrinks
```

We do **not** build the brain. `efficientnet_b0` is a famous brain that already learned to see objects by studying millions of internet photos. We only **fine-tune** it on your 3 classes (this is called *transfer learning*). That is why the whole thing trains in minutes, not days.

### The files, and their job (kitchen analogy)

| File | Real job | Analogy |
|---|---|---|
| `src/audit.py` | Counts photos, checks none are corrupted | Checking your ingredients before cooking |
| `src/train_resumable.py` | Loads data, trains the model, saves progress every epoch | The cooking + a camera that films progress |
| `configs/baseline.yaml` | The settings (epochs, image size, learning rate…) | The recipe card |
| `configs/profiles.yaml` | Settings per machine (lab GPU vs home CPU) | Who's doing the cooking and with what stove |
| `results/*.json` | The final scores (accuracy, F1…) | The taste-test report |


## Part 1 — The libraries (what each import is for)

When you see `import torch` in code, this table is what that line is buying you:

| Library | What it is | Why this project needs it |
|---|---|---|
| `torch` | The engine for neural networks + automatic math | Builds the brain, runs the training loop |
| `torchvision` | Video/vision helper built on torch | `ImageFolder` (reads folders-of-photos), `transforms` (resize/normalize) |
| `timm` | Database of many ready-made pretrained models | Gives us `efficientnet_b0` with ImageNet weights |
| `PIL` (Pillow) | Reads/writes image files | Opening `.jpg` files |
| `numpy` | Fast number arrays | Working with predictions/labels numbers |
| `matplotlib` | Plotting/graphs | Showing sample images, (later) curves |
| `scikit-learn` | Classic ML metrics | `accuracy`, F1, confusion matrix |
| `yaml` | Reads `.yaml` config files | Loading `baseline.yaml` settings |
| `tqdm` | Progress bars | The `[####] 45%` you see while training |
| `huggingface_hub` | Cloud storage (Hugging Face) | Uploading/downloading checkpoints between machines |
| `imagehash` | Perceptual image fingerprints | (Phase 2) finding near-duplicate photos across datasets |

**Rule of thumb:** you rarely need to know *how* these work internally — just *what job each one does* in our pipeline.


In [ ]:
# 1) Our toolbox — run me first
import torch, torchvision, timm
import PIL
import numpy as np
import sklearn
import yaml
import tqdm
print("torch        ", torch.__version__)
print("torchvision  ", torchvision.__version__)
print("timm         ", timm.__version__)
print("Pillow       ", PIL.__version__)
print("numpy        ", np.__version__)
print("scikit-learn ", sklearn.__version__)
print("yaml         ", yaml.__version__)


## Part 2 — Look at the data yourself (no magic)

The dataset is *already* split 70/15/15 and sorted into folders. Let's count it using plain Python — no library tricks.


In [ ]:
# 2) Count the photos with pure Python
from pathlib import Path

ROOT = Path(r"F:/Downloads/skin_disease_images")
print("Counting...")
grand_total = 0
for split in ["train", "validation", "test"]:
    d = ROOT / split
    per_class = {}
    for cls in sorted(p for p in d.iterdir() if p.is_dir()):
        n = sum(1 for f in cls.rglob("*") if f.is_file())
        per_class[cls.name] = n
    tot = sum(per_class.values())
    grand_total += tot
    print(f"{split:12s} total={tot:5d}  " + " | ".join(f"{k}:{v}" for k, v in per_class.items()))
print(f"---\nGRAND TOTAL = {grand_total}")


**You should see:** train=1706, validation=366, test=367, total=2439.

This is exactly what `src/audit.py` does — but it also checks each file opens correctly (corruption check) and saves everything to a JSON so the numbers are on record.


## Part 3 — See the photos

A photo is just a grid of colored numbers (255 x 255 x 3 channels). The model can't see it any other way. Let's look at one real photo from each class:


In [ ]:
# 3) Show one real photo per class
%matplotlib inline
import matplotlib.pyplot as plt
from PIL import Image
import glob

def first_image(cls):
    return sorted(glob.glob(str(ROOT / "train" / cls / "*")))[0]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, cls in zip(axes, ["acne", "normal", "rosacea"]):
    p = first_image(cls)
    img = Image.open(p)
    ax.imshow(img)
    ax.set_title(f"{cls}  ({img.size[0]}x{img.size[1]})")
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Notice the sizes differ -> the model will resize everything to 224x224.")


## Part 4 — `src/audit.py` explained

Open `src/audit.py` in your editor and follow along. It has 3 small ideas:

**1) `EXTS` + `list_images()`** — walk a folder tree and yield every image file:

```python
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def list_images(root):
    for p in root.rglob("*"):          # walk every file below root
        if p.is_file() and p.suffix.lower() in EXTS:
            yield p                     # hand it over one at a time
```
`rglob("*")` = "recursive glob = find everything under this folder". `suffix` = the file's extension. `yield` makes it a generator (lazy - doesn't load them all at once).

**2) `audit()`** — for each split folder it counts files per class, then (optionally) opens every file with `Image.open(...).verify()` to confirm it is not corrupted. Corrupt files would crash training later, so we find them now.

**3) `main()`** — the CLI:
- `--data` = dataset root  • `--check-corrupt` = scan every file (slower)  • `--out` = where to save the JSON

Run the real thing on your machine now (takes ~1 minute):


In [ ]:
# 4) Run the actual audit script, live
!python "F:/cse475_skin/src/audit.py" --data "F:/Downloads/skin_disease_images" --check-corrupt --out "F:/cse475_skin/results/audit_starter.json"


The output saves to `results/audit_starter.json`. You'll see class counts per split and — importantly — **`corrupt_files: []`** (0 broken images).

> Why do this step? If a photo is corrupt, training crashes 20 minutes in and you waste time. 1 minute of checking up front saves an hour later.


## Part 5 — `src/train_resumable.py` explained (the heart of the project)

This is the big file (~560 lines). Don't read it top-to-bottom — read it as 8 chunks. Each chunk below shows the real code plus what/why.

### Chunk 1 — CLI + machine detection

```python
ap.add_argument("--config")          # which config (recipe) to use
ap.add_argument("--profile")         # which machine profile
ap.add_argument("--device")          # cuda / dml / cpu
ap.add_argument("--data")            # override dataset root
ap.add_argument("--resume")          # start from a saved checkpoint? ("auto" = yes)
ap.add_argument("--hub")             # "local" or "hf" (cloud backup)
ap.add_argument("--hf-repo")         # your cloud checkpoint repo
ap.add_argument("--epochs")          # how many times to study the data
```

Then it *detects what machine it's on* (`detect_environment` + `pick_profile`):

| Machine | Detected by | Profile picked |
|---|---|---|
| Lab A4000 (CUDA) | GPU name contains "a4000" | `lab_a4000` |
| Kaggle | `/kaggle/working` folder exists | `kaggle_t4` |
| Colab | `/content/drive` exists | `colab_t4` |
| Home RX580 | `torch_directml` installed | `home_rx580_dml` |
| Anything else | fallback | `home_rx580_cpu` |

**Why?** The same script must run on 4 very different machines (CUDA vs DirectML vs plain CPU). The profile tells it: batch size, whether to use AMP, how many CPU workers. One command, four machines — this is the "portable training" superpower of the project.


### Chunk 2 — `build_loaders`: getting the data ready to eat

```python
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),          # model wants square 224x224
    transforms.RandomHorizontalFlip(),       # study-time trick #1: mirror some photos
    transforms.RandomRotation(15),           # study-time trick #2: tilt some photos
    transforms.ColorJitter(0.1, 0.1, 0.1),   # study-time trick #3: shift colours a bit
    transforms.ToTensor(),                   # bytes -> numbers a model can compute with
    transforms.Normalize([0.485,0.456,0.406],  # match the stats the pretrained model
                         [0.229,0.224,0.225]), # was trained on (mean/std scaling)
])
```

- **`ImageFolder(path)`** — torchvision's helper that treats each subfolder as a class. Folder `acne/` → label 0, `normal/` → label 1, etc.
- **The 3 "Random" tricks are *data augmentation***: we deliberately give the model bent/mirrored photos during study so it learns "a pimple is still a pimple when flipped". The validation/test sets get **no** augmentation (we want to test fairly).
- **`DataLoader(ds, batch_size=32, shuffle=True)`** — feeds photos to training in groups ("batches") of 32, in random order. `num_workers=4` lets 4 CPU threads load photos in parallel so the GPU isn't idle.
- **`shuffle=True` on train / `False` on val/test** — during study, order matters for learning; during testing, order doesn't matter.

Let's see a transform in action:


In [ ]:
# 5) What the model actually sees after a transform
from torchvision import transforms

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

t = tf(Image.open(first_image("normal")))
print("A photo becomes a tensor of shape:", tuple(t.shape))   # (3, 224, 224)
print("-> 3 colour channels, 224 wide, 224 tall")
print("Value range:", float(t.min()), "to", float(t.max()))
print("The Normalize step rescales pixels to ~mean 0, std 1 (instead of 0..255).")


In [ ]:
# 6) Augmentation demo: 6 versions of the SAME photo
# (training would see all these during study -> model becomes robust)
fig, axes = plt.subplots(2, 3, figsize=(11, 6))
aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3, 0.3, 0.3),
    transforms.ToTensor(),
])
tf1 = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()])
base = tf1(Image.open(first_image("rosacea"))).permute(1,2,0)
for i, ax in enumerate(axes.flat):
    if i == 0:
        show = base
    else:
        show = aug(Image.open(first_image("rosacea"))).permute(1,2,0)
    ax.imshow(show.clamp(0,1))
    ax.set_title("original" if i == 0 else "augmented")
    ax.axis("off")
plt.tight_layout(); plt.show()
print("Same photo, 6 looks - the model trains on all of them.")


### Chunk 3 — The brain (model), loss, optimizer, scheduler

```python
model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=3)
criterion = nn.CrossEntropyLoss()                                   # how wrong is the guess
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs*steps)
```

- **`timm.create_model(..., pretrained=True)`** — downloads the brain pre-trained on ImageNet, swaps the final "reads out 1000 object types" layer for one that reads out *your 3 classes*. Then we fine-tune.
- **`CrossEntropyLoss`** — if the photo is really "acne" and the model says 70% acne / 20% normal / 10% rosacea, the loss is small; if it says 90% rosacea, the loss is big. Training = pushing this number down.
- **`AdamW`** — the study method. Uses `lr` (learning rate) = step size for nudging wires. `weight_decay` is a tiny "forget old behaviour" pressure so it doesn't overfit.
- **`CosineAnnealingLR`** — the learning-rate schedule: start stepping boldly, then take smaller and smaller steps as training goes on (like "revise hard early, fine-tune later"). T_max = how many training steps the cycle spans.

Quick sanity check that the brain works:


In [ ]:
# 7) Load the brain, feed it a fake photo, see the guess
model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=3)
n = sum(p.numel() for p in model.parameters())
print(f"EfficientNet-B0 has {n/1e6:.1f} million adjustable wires (parameters)")
model.eval()

fake_photos = torch.randn(2, 3, 224, 224)   # 2 nonsense images (just random numbers)
with torch.no_grad():
    out = model(fake_photos)
print("Input : batch of", fake_photos.shape[0], "photos")
print("Output:", tuple(out.shape), "-> one score per class per photo")
print("Score per photo (acne/normal/rosacea):", out.tolist())
print("Pick the biggest score per photo (argmax):", out.argmax(1).tolist())
print("Because the input is random, the guesses are meaningless - fine. We'll train next.")


### Chunk 4 — The training loop (the 6 sacred lines)

```python
for x, y in loader:                      # one batch of 32 photos + labels
    optimizer.zero_grad()                # clear last batch's nudges
    out = model(x)                       # guess
    loss = criterion(out, y)             # how wrong
    loss.backward()                      # compute gradient (which wires caused it)
    optimizer.step()                     # nudge the wires
```

That's the whole secret. `run_epoch(...)` in the file wraps this loop and also counts:
- `correct / total` → the accuracy of that epoch
- `loss_sum / total` → the average loss (should go down over epochs)

Then an **evaluation pass** over the validation set (with `model.eval()` + `torch.no_grad()` so it only measures, doesn't learn). The results of all 15 epochs are saved into `history`.

**Scheduler timing:** `scheduler.step()` is called once per epoch, which shrinks the learning rate over time.


### Chunk 5 — Checkpoints + resume (nothing is ever lost)

```python
torch.save({
    "model_state":      ...   # the brain
    "optimizer_state":  ...   # the study-momentum (AdamW keeps stats!)
    "scheduler_state":  ...   # where we are on the cosine curve
    "epoch":            ...   # how far we got
    "history":          ...
    "rng_*":            ...   # random-number states so shuffling continues identically
}, "results/<run>_last.pt")
```

- `_last.pt` = every epoch's position → used to **resume** after a crash or a machine switch.
- `_best.pt` = the exact moment validation accuracy was highest → used as the final model.
- The `configs/profiles.yaml` decides which folder they land in (e.g. `/kaggle/working/results` on Kaggle).

**Why save optimizer + RNG too?** AdamW remembers momentum; the DataLoader shuffles randomly. Resume correctly re-creates the *same* machine state, not just the weights — that's what makes a true "pick up where we left off" work (that's Section 11 of `AGENT.md`, the portability design).


### Chunk 6 — Cloud sync (Hugging Face)

```python
maybe_download_from_hf("Nirob-jon/cse475-skin-checkpoints", "phase1_baseline_last.pt", ...)
maybe_upload_to_hf(...)   # called after every epoch
```

- **Start:** if `--hub hf --resume auto` and no local checkpoint, download the latest from HF.
- **Every epoch:** upload the newest `_last.pt`.
- **Result:** you can start training on the lab, continue on Kaggle, finish on Colab — the checkpoint follows you. (Bangladesh power cuts, 30 hr/week Kaggle quota and 3 different machines make this a genuine lifesaver.)


### Chunk 7 — The final exam (test) + metrics

```python
_, test_acc, preds, labels = evaluate(model, dl_test, criterion, device)
rep      = classification_report(labels, preds, target_names=classes, ...)
cm       = confusion_matrix(labels, preds)
macro_f1 = f1_score(labels, preds, average="macro")
```

- The **test** set was never shown during training or validation — it's the honest final exam.
- **accuracy** = times right ÷ total.
- **macro F1** = average of per-class F1; punishes you if one class is bad (better than accuracy for imbalanced sets like this one).
- **confusion matrix** = a grid: rows are true class, columns are predicted class. Diagonal = correct.

This JSON is your **Phase 1 deliverable** and the golden-rule gate for opening the second dataset.


### Chunk 8 — Now watch it actually train (1 epoch, your CPU, ~3 min)

We run the *exact same script* used for the real Phase 1, but for **1 epoch** and with a separate run name (`tutorial_demo`) so `results/phase1_baseline.json` is never touched.


In [ ]:
# 8) A real 1-epoch training run on your home PC (CPU)
!python "F:/cse475_skin/src/train_resumable.py" --config "F:/cse475_skin/configs/tutorial_cpu.yaml" --data "F:/Downloads/skin_disease_images" --profile home_rx580_cpu --epochs 1


While it runs, notice: `[RESUME] no checkpoint` (fresh start), `[AMP] False` (CPU has no mixed precision), the tqdm progress bar, then `[EPOCH 1]` with `train_acc`/`val_acc`, then the final `[DONE] ... test_acc=...` line that wrote the JSON.

Read the result:


In [ ]:
# 9) Read the 1-epoch result
import json
r = json.load(open(r"F:/cse475_skin/results/tutorial_demo.json"))
print("test_acc  =", r["test_acc"])
print("macro_f1  =", r["macro_f1"])
print("\nconfusion matrix (rows = true, cols = predicted):")
for name, row in zip(r["classes"], r["confusion_matrix"]):
    print(f"  {name:9s}", row)

# cleanup so results/ stays clean for the real run
# (uncomment to run when you're done reading):
# import os, glob
# for f in glob.glob(r"F:/cse475_skin/results/tutorial_demo*"):
#     os.remove(f)


## Part 6 — `src/train.py` (the older twin — removed)

There used to be a second, simpler trainer here called `src/train.py`. It has been **deleted** to remove the confusion you noticed — it was never used by the project.

**Today there is exactly ONE trainer: `src/train_resumable.py`.** If you ever see `train.py` mentioned in old notes or session-log history, mentally translate it to `train_resumable.py`.

A permanent one-page map of every file (what to run, what to ignore) now lives at `docs/FILE_MAP.md`.


## Part 7 — The configs (recipe cards)

### `configs/baseline.yaml` — every key means something

| Key | Value | What it does | Raising/lowering it? |
|---|---|---|---|
| `data_root` | `F:/Downloads/skin_disease_images` | Where the photos are | Point at another dataset |
| `model` | `efficientnet_b0` | Which brain | `efficientnet_b3` = bigger/slower, better |
| `epochs` | `15` | Study sessions | More = longer/better… or overfit |
| `batch_size` | `32` | Photos per batch | Smaller fits low memory, noisier gradients |
| `img_size` | `224` | Resize target | 384 higher quality, 4x slower |
| `lr` | `0.001` | Study step size | Too big = unstable, too small = slow |
| `weight_decay` | `0.0001` | Anti-overfitting pressure | 0 = prone to overfit |
| `seed` | `42` | Randomness lock | Keep 42 = you can reproduce a run |
| `run_id` | `phase1_baseline` | Names all output files | `results/phase1_baseline*.json/.pt` |
| `results_dir` | `F:/cse475_skin/results` | Where scores/checkpoints go | — |

### `configs/profiles.yaml` — one row of equipment per machine

| Profile | device | batch | AMP | Where checkpoints go |
|---|---|---|---|---|
| `lab_a4000` | cuda | 32 | on | project `results/` |
| `kaggle_t4` | cuda | 32 | on | `/kaggle/working/results` |
| `colab_t4` | cuda | 32 | on | `/content/drive/MyDrive/cse475/results` |
| `home_rx580_dml` | dml | 16 | off | project `results/` |
| `home_rx580_cpu` | cpu | 8 | off | project `results/` |

**AMP** (automatic mixed precision) = run part of the math in a faster half-precision format — available only on CUDA. Notice GPU profiles are uniform (batch 32) so results stay comparable across machines.


## Part 8 — What literally happens when you press "Run all" on Kaggle

Your `notebooks/kaggle_phase1.ipynb` cell-by-cell:

| Cell | What runs | Why |
|---|---|---|
| 1 | sign in to HF with your token (secret or pasted) | Needed to download checkpoints/upload them |
| 2 | `git clone` the public repo | Pulls all `src/`, `configs/`, `requirements.txt` onto Kaggle |
| 3 | download the dataset from HF | 72 MB → appears in the kernel as a folder |
| 4 | `pip install timm tqdm pyyaml imagehash ...` | torch/torchvision already preinstalled on Kaggle |
| 5 | run `audit.py` | same sanity check you did on your PC |
| 6 | show 3 sample images | visual proof the data looks right |
| 7 | run `train_resumable.py --profile kaggle_t4 --hub hf --resume auto` | the real training (T4 GPU, ~11 min) |
| 8 | print `test_acc` + confusion matrix | the Phase 1 number you're after |

The checkpoint is pushed to HF after **every epoch**, so if you only have 2 hours today on Kaggle and 30 min tomorrow on Colab, `--resume auto` picks up exactly where you left off.


## Part 9 — How to read your results

From the smoke-test run we did earlier (2 epochs, CPU) — real numbers, same pipeline:

```
test_acc   = 0.8747   -> 87% of test photos classified correctly after just 2 epochs
macro_f1   = 0.8617   -> per-class F1 averaged (punishes unbalanced performance)
confusion matrix:
  acne    [73 26  0]   73 acne right, 26 mislabeled as normal, 0 as rosacea
  normal  [ 4 97  2]   most normals right, 4 called acne
  rosacea [ 2 12 151]  151 rosacea right
```

Reading a confusion matrix: **rows = truth, columns = prediction**.
- The diagonal (73, 97, 151) = correct.
- Off-diagonal = specific mistakes. Here `acne → normal` (26) is the biggest error — that's where to study for the paper's error analysis.
- With 15 epochs on the lab A4000, expect `test_acc` comfortably above that.

Why both accuracy **and** F1? The set is imbalanced (rosacea has 770, acne only 460). Accuracy can hide "I'm great at the common class, terrible at the rare one"; macro F1 averages the classes equally and catches that.


## Part 10 — The 15 phrases you'll keep hearing

| Term | 1-line meaning |
|---|---|
| **Epoch** | One complete pass over all training photos |
| **Batch** | The handful of photos the brain sees per update (32) |
| **Dataloader** | The waiter that feeds batches in order |
| **ImageFolder** | "Each subfolder is a class" reader |
| **Transform** | Pre-processing (resize, flip, normalize) |
| **Augmentation** | Random distortions during training that make the model robust |
| **Model / network** | The brain (a bunch of numbers = weights) |
| **Parameter / weight** | One adjustable number inside the brain (B0 has ~5.3M) |
| **Logits / output** | Raw class scores before turning into probabilities |
| **argmax** | "Pick the biggest score" → the predicted class |
| **Loss** | How wrong the guess was (bigger = worse) |
| **backward / gradient** | Which wires were responsible for the mistake |
| **Optimizer (AdamW)** | How the wires get nudged (with `lr` step size) |
| **Scheduler (cosine)** | Shrinks `lr` over time |
| **Checkpoint** | A saved snapshot of everything, so you can resume |
| **Transfer learning** | Start from a brain pre-trained on ImageNet, fine-tune it |
| **overfit** | Memorizes training photos, fails on new ones (watch train_acc >> val_acc) |

## You do NOT need to memorize any of this

You need to be able to *explain* three sentences at your defense:

1. "I fine-tuned a pre-trained EfficientNet-B0 with transfer learning."
2. "I trained for 15 epochs, batch 32, saving checkpoints every epoch to resume anywhere."
3. "I measured accuracy, macro F1, and a confusion matrix on a held-out test set."

Congratulations — after this notebook you can say all three honestly. 🎓


## Next steps (your real Phase 1)

1. **This tutorial** — done ✅
2. **Lab A4000** (or Kaggle/Colab) — run `scripts/lab_phase1.ps1` or the notebook → get `results/phase1_baseline.json`
3. Send the numbers to me, update the session log in `AGENT.md`, then Phase 2 (union + harmonize + dedupe) can start.

Any question while reading — copy the cell output and paste it here. I'll explain it in plain English.
